# Z5 three-case comparison — in-house solver vs PyNite

This notebook is dedicated to the three Z5 load cases below. It runs the **same Z5 geometry and section data** through two independent analysis paths:

1. the project's in-house mixed frame/truss solver;
2. the PyNite adapter.

The main engineering question is:

> **Which elements are more highly loaded in Case 1 than in both Case 2 and Case 3?**

The governing-element check deliberately avoids comparing raw signed stress numerically. Instead it compares physically meaningful demand measures:

- absolute axial force;
- compression magnitude;
- tension magnitude;
- absolute combined normal stress;
- bending resultant.

Positive axial force/stress means **tension** and negative means **compression**.

For direct solver-to-solver comparison this notebook uses:

```python
SECTION_MODE = "isotropic"
CROSSING_DIAGONAL_MODE = "continuous"
```

These are the recommended comparison settings from `pynite_adapter.py`, because they reduce local-axis sensitivity and avoid the connected-crossing stabilization difference between the two backends.

## Modelling assumptions and joint idealization

This comparison uses the existing **mixed frame/truss idealization** of the Z5 model. The distinction between frame and truss members is important when interpreting the results:

- **Frame elements are treated as rigidly connected at their nodes unless an explicit release exists.** In the in-house solver they are 3D Euler-Bernoulli frame elements with shared translational and rotational DOFs, so connected frame members can transfer axial force, shear, torsion and bending moments through the node. In practical terms, these joints are idealized as **fixed/rigid**, not pinned.
- **Truss elements are axial-only.** In the in-house solver they have no bending or torsional stiffness, so their end connections behave as **pins for member-force purposes**: they transmit axial force only.
- In the **PyNite comparison**, frame members are ordinary PyNite 3D members with no end releases. CSV truss members are approximated as pin-ended PyNite members by releasing local bending rotations `Ry` and `Rz` at both ends. This closely reproduces the intended axial-bracing behaviour, but it is not mathematically identical to the in-house true truss formulation.
- The analysis uses `CROSSING_DIAGONAL_MODE = "continuous"`. At a geometric crossing of two diagonals, the crossing is **not treated as a physical joint between the diagonals**. The split input segments are merged back into continuous diagonal members for analysis.
- Nodes marked as supports, together with the minimum-Z support nodes under the current solver settings, are restrained as fixed supports.
- No connection eccentricity, bolt/gusset flexibility, local joint slip, member imperfections, or connection semi-rigidity is included. The model is therefore an idealized linear-elastic load-distribution model rather than a detailed connection model.

These assumptions are applied consistently across all three load cases. The PyNite path is intended as an independent numerical cross-check of the same structural idealization, not as a different connection model.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root() -> Path:
    """Find the repository root whether the notebook starts in /notebooks or the repo root."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "solver.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the keratia project root containing solver.py and data/.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "z5_three_case_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from solver import read_model, SolveOptions, solve_model, summarize_result
from helpers import plot_signed_stress_3d, plot_deformed_shape
from pynite_adapter import PyNiteRunOptions, analyze_pynite_model, summarize_pynite_result

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

## 1. Inputs and analysis settings

Each load row is applied to one of the two Z5 high load points. Units are N and N·mm.

In [ ]:
SECTION_MODE = "isotropic"
PRINCIPAL_ORIENTATION = "strong_inward"
CROSSING_DIAGONAL_MODE = "continuous"

DEFORMED_SHAPE_SCALE = 100.0

Z5_CASES = {
    "Case_1": [
        {"node_label": "high_xneg", "Fx": 16250.0, "Fy": 8320.0, "Fz": -20000.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 16250.0, "Fy": 8320.0, "Fz": -20000.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
    "Case_2": [
        {"node_label": "high_xneg", "Fx": 48900.0, "Fy": 0.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 48900.0, "Fy": 0.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
    "Case_3": [
        {"node_label": "high_xneg", "Fx": 21600.0, "Fy": 38000.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 21600.0, "Fy": 38000.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
}

load_rows = []
for case_name, rows in Z5_CASES.items():
    for row in rows:
        load_rows.append({"case": case_name, **row})

display(pd.DataFrame(load_rows))

In [ ]:
z5_model = read_model(
    "Z5",
    DATA_DIR / "Z5_nodes_aligned.csv",
    DATA_DIR / "Z5_elements_with_sections.csv",
)

print("Input Z5 nodes:", len(z5_model.nodes))
print("Input Z5 elements:", len(z5_model.elements))
display(z5_model.elements["element_type"].value_counts().rename("count").to_frame())

## 2. Shared comparison helpers

The dedicated Case-1-governing table uses the following derived demands:

- `abs_axial_kN`: absolute axial-force magnitude;
- `compression_kN`: compression magnitude only, zero in tension;
- `tension_kN`: tension magnitude only, zero in compression;
- `sigma_abs_MPa`: `sigma_max_abs_MPa`;
- `bending_kNmm`: maximum bending resultant.

An element is reported only when **Case 1 is strictly greater than both Case 2 and Case 3** for at least one of those measures.

In [ ]:
def stacked_member_results(case_results, backend_name):
    """Stack the complete member-force/result table for every case."""
    frames = []
    for case_name, result in case_results.items():
        table = result["member_forces"].copy()
        table.insert(0, "case", case_name)
        table.insert(0, "backend", backend_name)
        frames.append(table)
    return pd.concat(frames, ignore_index=True).sort_values(["case", "element_id"]).reset_index(drop=True)


def _strictly_greater_than_both(c1, c2, c3, rtol=1e-7, atol=1e-9):
    """Avoid classifying floating-point equality as a real exceedance."""
    other_max = max(float(c2), float(c3))
    c1 = float(c1)
    return (c1 > other_max) and (not np.isclose(c1, other_max, rtol=rtol, atol=atol))


def case1_governing_elements(case_results):
    """
    Return only elements where Case 1 exceeds BOTH Case 2 and Case 3
    in at least one physically meaningful demand measure.
    """
    required = {"Case_1", "Case_2", "Case_3"}
    missing = required - set(case_results)
    if missing:
        raise KeyError(f"Missing cases: {sorted(missing)}")

    indexed = {
        case: result["member_forces"].set_index("element_id")
        for case, result in case_results.items()
    }

    common_ids = sorted(
        set(indexed["Case_1"].index)
        & set(indexed["Case_2"].index)
        & set(indexed["Case_3"].index)
    )

    rows = []
    for eid in common_ids:
        r1 = indexed["Case_1"].loc[eid]
        r2 = indexed["Case_2"].loc[eid]
        r3 = indexed["Case_3"].loc[eid]

        def demand_values(r):
            N = float(r["axial_force_signed_N"])
            return {
                "abs_axial_kN": float(r["max_abs_N_N"]) / 1000.0,
                "compression_kN": max(-N, 0.0) / 1000.0,
                "tension_kN": max(N, 0.0) / 1000.0,
                "sigma_abs_MPa": float(r["sigma_max_abs_MPa"]),
                "bending_kNmm": float(r["max_M_resultant_Nmm"]) / 1000.0,
            }

        d1, d2, d3 = demand_values(r1), demand_values(r2), demand_values(r3)

        governing_metrics = [
            metric
            for metric in d1
            if _strictly_greater_than_both(d1[metric], d2[metric], d3[metric])
        ]
        if not governing_metrics:
            continue

        pct_margins = []
        for metric in governing_metrics:
            next_worst = max(d2[metric], d3[metric])
            if abs(next_worst) > 1e-12:
                pct_margins.append((d1[metric] - next_worst) / abs(next_worst) * 100.0)
            elif d1[metric] > 0:
                pct_margins.append(np.inf)

        rows.append({
            "element_id": int(eid),
            "element_type": r1.get("element_type", ""),
            "start_label": r1.get("start_label", ""),
            "end_label": r1.get("end_label", ""),
            "member_role": r1.get("member_role", ""),
            "section_name": r1.get("section_name", ""),
            "governing_metrics": ", ".join(governing_metrics),
            "largest_C1_margin_pct": max(pct_margins) if pct_margins else np.nan,

            "C1_axial_signed_kN": float(r1["axial_force_signed_N"]) / 1000.0,
            "C2_axial_signed_kN": float(r2["axial_force_signed_N"]) / 1000.0,
            "C3_axial_signed_kN": float(r3["axial_force_signed_N"]) / 1000.0,

            "C1_abs_axial_kN": d1["abs_axial_kN"],
            "C2_abs_axial_kN": d2["abs_axial_kN"],
            "C3_abs_axial_kN": d3["abs_axial_kN"],

            "C1_compression_kN": d1["compression_kN"],
            "C2_compression_kN": d2["compression_kN"],
            "C3_compression_kN": d3["compression_kN"],

            "C1_tension_kN": d1["tension_kN"],
            "C2_tension_kN": d2["tension_kN"],
            "C3_tension_kN": d3["tension_kN"],

            "C1_sigma_abs_MPa": d1["sigma_abs_MPa"],
            "C2_sigma_abs_MPa": d2["sigma_abs_MPa"],
            "C3_sigma_abs_MPa": d3["sigma_abs_MPa"],

            "C1_bending_kNmm": d1["bending_kNmm"],
            "C2_bending_kNmm": d2["bending_kNmm"],
            "C3_bending_kNmm": d3["bending_kNmm"],
        })

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(["largest_C1_margin_pct", "element_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

## 3. In-house solver

This section runs the three cases through `solver.py`.

In [ ]:
inhouse_options = SolveOptions(
    crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
    section_mode=SECTION_MODE,
    principal_orientation=PRINCIPAL_ORIENTATION,
)

z5_inhouse_results = {}
for case_name, loads in Z5_CASES.items():
    print(f"Solving Z5 {case_name} with in-house solver...")
    z5_inhouse_results[case_name] = solve_model(z5_model, loads, inhouse_options)

inhouse_summary = pd.DataFrame([
    {"case": case_name, **summarize_result(result)}
    for case_name, result in z5_inhouse_results.items()
])

display(inhouse_summary)

### 3.1 Case 1 worse than both Case 2 and Case 3 — in-house

Only elements that satisfy the strict three-case governing rule appear below.

In [ ]:
inhouse_case1_governing = case1_governing_elements(z5_inhouse_results)

print("In-house elements where Case 1 governs over BOTH Case 2 and Case 3:")
print("Count:", len(inhouse_case1_governing))
display(inhouse_case1_governing)

### 3.2 All in-house element loads/results

This is the complete `member_forces` output for every element in every case. No elements are filtered out.

In [ ]:
inhouse_all_element_results = stacked_member_results(z5_inhouse_results, "in_house")
display(inhouse_all_element_results)

## 4. PyNite approach

This section runs the exact same three load cases through the project's `pynite_adapter.py`.

The adapter represents CSV truss members as pin-ended PyNite members with local `Ry/Rz` releases, so this is an independent comparison route rather than a bit-for-bit duplicate of the in-house truss formulation.

In [ ]:
pynite_options = PyNiteRunOptions(
    crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
    section_mode=SECTION_MODE,
    principal_orientation=PRINCIPAL_ORIENTATION,
)

z5_pynite_results = {}
for case_name, loads in Z5_CASES.items():
    print(f"Solving Z5 {case_name} with PyNite...")
    z5_pynite_results[case_name] = analyze_pynite_model(
        z5_model,
        loads,
        options=pynite_options,
        case_name=case_name,
        combo_name=case_name,
    )

pynite_summary = pd.DataFrame([
    {"case": case_name, **summarize_pynite_result(result)}
    for case_name, result in z5_pynite_results.items()
])

display(pynite_summary)

### 4.1 Case 1 worse than both Case 2 and Case 3 — PyNite

The same governing criteria used for the in-house solver are applied here.

In [ ]:
pynite_case1_governing = case1_governing_elements(z5_pynite_results)

print("PyNite elements where Case 1 governs over BOTH Case 2 and Case 3:")
print("Count:", len(pynite_case1_governing))
display(pynite_case1_governing)

### 4.2 All PyNite element loads/results

This is the complete member-results table returned by the current PyNite adapter for all elements and all three cases.

In [ ]:
pynite_all_element_results = stacked_member_results(z5_pynite_results, "PyNite")
display(pynite_all_element_results)

## 5. Direct comparison of the Case-1-governing element sets

This section makes it easy to see whether the two solver paths identify the same elements as being more highly loaded in Case 1 than in **both** alternatives.

In [ ]:
inhouse_ids = set(inhouse_case1_governing["element_id"]) if not inhouse_case1_governing.empty else set()
pynite_ids = set(pynite_case1_governing["element_id"]) if not pynite_case1_governing.empty else set()
all_governing_ids = sorted(inhouse_ids | pynite_ids)

comparison_rows = []
for eid in all_governing_ids:
    ih = inhouse_case1_governing.loc[inhouse_case1_governing["element_id"] == eid]
    py = pynite_case1_governing.loc[pynite_case1_governing["element_id"] == eid]

    source = ih.iloc[0] if not ih.empty else py.iloc[0]
    comparison_rows.append({
        "element_id": eid,
        "element_type": source.get("element_type", ""),
        "start_label": source.get("start_label", ""),
        "end_label": source.get("end_label", ""),
        "member_role": source.get("member_role", ""),
        "section_name": source.get("section_name", ""),
        "inhouse_C1_governs": eid in inhouse_ids,
        "pynite_C1_governs": eid in pynite_ids,
        "inhouse_governing_metrics": "" if ih.empty else ih.iloc[0]["governing_metrics"],
        "pynite_governing_metrics": "" if py.empty else py.iloc[0]["governing_metrics"],
    })

governing_set_comparison = pd.DataFrame(comparison_rows)
display(governing_set_comparison)

print("Common governing elements:", sorted(inhouse_ids & pynite_ids))
print("In-house only:", sorted(inhouse_ids - pynite_ids))
print("PyNite only:", sorted(pynite_ids - inhouse_ids))

## 6. Save comparison tables

The notebook also writes the principal tables to `results/z5_three_case_comparison/` so they can be inspected without re-running the display cells.

In [ ]:
inhouse_summary.to_csv(RESULTS_DIR / "inhouse_summary.csv", index=False)
pynite_summary.to_csv(RESULTS_DIR / "pynite_summary.csv", index=False)

inhouse_all_element_results.to_csv(
    RESULTS_DIR / "inhouse_all_element_results.csv", index=False
)
pynite_all_element_results.to_csv(
    RESULTS_DIR / "pynite_all_element_results.csv", index=False
)

inhouse_case1_governing.to_csv(
    RESULTS_DIR / "inhouse_case1_governs_both.csv", index=False
)
pynite_case1_governing.to_csv(
    RESULTS_DIR / "pynite_case1_governs_both.csv", index=False
)
governing_set_comparison.to_csv(
    RESULTS_DIR / "case1_governing_set_backend_comparison.csv", index=False
)

print("Wrote comparison CSVs to:", RESULTS_DIR)

## 7. Whole-construction tension/compression visualizations

The plots below use **axial signed stress** so that the color directly represents the axial load path:

- positive = tension;
- negative = compression.

Each figure is autoscaled symmetrically around zero by the existing plotting helper. Therefore use the numerical tables above—not color intensity alone—for quantitative comparison between different cases.

In [ ]:
for case_name, result in z5_inhouse_results.items():
    fig, ax = plot_signed_stress_3d(
        result["nodes"],
        result["elements"],
        result["member_forces"],
        title=f"Z5 in-house axial tension/compression — {case_name}",
        stress_column="sigma_axial_signed_MPa",
        annotate_load_points=True,
        annotate_elements=False,
        show_frame=True,
        show_truss=True,
    )
    plt.show()

In [ ]:
for case_name, result in z5_pynite_results.items():
    fig, ax = plot_signed_stress_3d(
        result["nodes"],
        result["elements"],
        result["member_forces"],
        title=f"Z5 PyNite axial tension/compression — {case_name}",
        stress_column="sigma_axial_signed_MPa",
        annotate_load_points=True,
        annotate_elements=False,
        show_frame=True,
        show_truss=True,
    )
    plt.show()

### Optional: signed controlling combined stress

For frame members, `sigma_extreme_signed_MPa` includes axial force plus bending at the controlling extreme fiber. For trusses it remains axial stress. This is useful for seeing where the *controlling normal stress* is tensile or compressive, but it should not be confused with the pure axial state used for buckling screening.

In [ ]:
for backend_name, case_results in [
    ("in-house", z5_inhouse_results),
    ("PyNite", z5_pynite_results),
]:
    for case_name, result in case_results.items():
        fig, ax = plot_signed_stress_3d(
            result["nodes"],
            result["elements"],
            result["member_forces"],
            title=f"Z5 {backend_name} controlling signed normal stress — {case_name}",
            stress_column="sigma_extreme_signed_MPa",
            annotate_load_points=True,
            annotate_elements=False,
            show_frame=True,
            show_truss=True,
        )
        plt.show()

## 8. Whole-construction deformation visualizations

Undeformed geometry is shown faintly and the deformed shape is exaggerated by `DEFORMED_SHAPE_SCALE` for visibility.

In [ ]:
for case_name, result in z5_inhouse_results.items():
    fig, ax = plot_deformed_shape(
        result["nodes"],
        result["elements"],
        result["displacements"],
        scale=DEFORMED_SHAPE_SCALE,
        title=f"Z5 in-house deformed shape — {case_name}",
        show_truss=True,
    )
    plt.show()

In [ ]:
for case_name, result in z5_pynite_results.items():
    fig, ax = plot_deformed_shape(
        result["nodes"],
        result["elements"],
        result["displacements"],
        scale=DEFORMED_SHAPE_SCALE,
        title=f"Z5 PyNite deformed shape — {case_name}",
        show_truss=True,
    )
    plt.show()

## 9. Interpretation notes

When deciding whether Case 1 is more severe for a member:

- use `compression_kN` for compression/buckling screening;
- use `tension_kN` for tensile axial demand;
- use `sigma_abs_MPa` for the current simplified combined normal-stress comparison;
- use `bending_kNmm` for bending demand.

Do **not** interpret a raw signed value such as `-10 MPa > -80 MPa` as Case 1 being worse. In compression, `-80 MPa` is the larger compressive demand.

This notebook is still an exploratory linear-elastic model comparison. It does not by itself constitute a code buckling/capacity check.